In [2]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.inspection import permutation_importance


class FeatureSelector:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.rankings = {}
    
    def compute_correlations(self):
        correlations = {}
        for col in self.X.columns:
            r, _ = pearsonr(self.X[col], self.y)
            correlations[col] = abs(r)
        
        self.rankings['correlation'] = pd.Series(correlations).sort_values(ascending=False)
        return self.rankings['correlation']
    
    def compute_mlp_importance(self, hidden_layers=(50, 30)):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(self.X)
        
        mlp = MLPRegressor(
            hidden_layer_sizes=hidden_layers,
            max_iter=1000,
            random_state=42,
            early_stopping=True
        )
        mlp.fit(X_scaled, self.y)
        
        importance = self._gevrey_weights(mlp)
        
        self.rankings['mlp_gevrey'] = pd.Series(
            importance, index=self.X.columns
        ).sort_values(ascending=False)
        return self.rankings['mlp_gevrey']
    
    def _gevrey_weights(self, mlp):
        W = mlp.coefs_[0]
        V = mlp.coefs_[1]
        
        ni, nh = W.shape
        
        P = np.zeros((ni, nh))
        for h in range(nh):
            for i in range(ni):
                P[i, h] = abs(W[i, h]) * abs(V[h, 0])
        
        Q = np.zeros((ni, nh))
        for h in range(nh):
            denom = np.sum(P[:, h])
            if denom == 0:
                continue
            for i in range(ni):
                Q[i, h] = P[i, h] / denom
        
        Q = np.nan_to_num(Q)
        
        S = np.zeros(ni)
        for i in range(ni):
            for h in range(nh):
                S[i] += Q[i, h]
        
        RI = (S / S.sum()) * 100
        
        return RI
    
    def compute_rf_importance(self, n_estimators=100):
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(self.X, self.y)
        
        purity_importance = pd.Series(
            rf.feature_importances_, index=self.X.columns
        )
        self.rankings['rf_purity'] = purity_importance.sort_values(ascending=False)
        
        perm_result = permutation_importance(
            rf, self.X, self.y,
            n_repeats=10,
            random_state=42
        )
        accuracy_importance = pd.Series(
            perm_result.importances_mean, index=self.X.columns
        )
        self.rankings['rf_accuracy'] = accuracy_importance.sort_values(ascending=False)
        
        return self.rankings['rf_purity'], self.rankings['rf_accuracy']
    
    def aggregate_rankings(self):
        rank_df = pd.DataFrame()
        for name, ranking in self.rankings.items():
            rank_df[name] = ranking.rank(ascending=False)
        
        rank_df['mean_rank'] = rank_df.mean(axis=1)
        
        return rank_df.sort_values('mean_rank')
    
    def _compute_relevance(self):
        relevance = pd.DataFrame()
        for name, ranking in self.rankings.items():
            values = ranking.values.reshape(-1, 1)
            normalized = MinMaxScaler().fit_transform(values).flatten()
            relevance[name] = pd.Series(normalized, index=ranking.index)
        
        return relevance.mean(axis=1)
    
    def mrmr_selection(self, n_features=20):
        relevance = self._compute_relevance()
        
        selected = []
        remaining = list(self.X.columns)
        
        for _ in range(n_features):
            if not remaining:
                break
            
            if not selected:
                best = max(remaining, key=lambda f: relevance[f])
            else:
                scores = {}
                for f in remaining:
                    rel = relevance[f]
                    red = np.mean([abs(self.X[f].corr(self.X[s])) for s in selected])
                    scores[f] = rel - red
                best = max(scores, key=scores.get)
            
            selected.append(best)
            remaining.remove(best)
        
        return selected
    
    def run_full_selection(self, n_features=20):
        print("(i) Calculando correlações...")
        self.compute_correlations()
        
        print("(ii) Treinando MLP e calculando Gevrey weights...")
        self.compute_mlp_importance()
        
        print("(iii) Treinando Random Forest...")
        self.compute_rf_importance()
        
        print("\nAgregando rankings...")
        combined = self.aggregate_rankings()
        
        print(f"\nSelecionando {n_features} features com mRMR...")
        selected = self.mrmr_selection(n_features)
        
        return selected, combined


if __name__ == "__main__":
    X = pd.read_csv("../dataset/j_kampe.csv")
    y = pd.read_csv("../dataset/distances.csv")["distance"]

    X = X[1000:]
    y = y[1000:]
    
    selector = FeatureSelector(X, y)
    selected_features, rankings = selector.run_full_selection(n_features=20)
    
    print("\nFeatures selecionadas:")
    for i, feat in enumerate(selected_features, 1):
        print(f"  {i}. {feat}")
    
    print("\nRankings combinados:")
    print(rankings.head(20))
    
    result_df = pd.DataFrame({"features": selected_features})
    result_df.to_csv("../results/important_features_v2.csv", index=False)
    print("\nSalvo em ../results/important_features_v2.csv")

(i) Calculando correlações...
(ii) Treinando MLP e calculando Gevrey weights...
(iii) Treinando Random Forest...

Agregando rankings...

Selecionando 20 features com mRMR...

Features selecionadas:
  1. z_cogram
  2. d_lag_1
  3. z_gram
  4. d_lag_3
  5. d_lag_20
  6. d_lag_4
  7. d_lag_5
  8. d_lag_19
  9. d_lag_6
  10. d_lag_18
  11. z_cogram_lag_1
  12. d_lag_21
  13. d_lag_2
  14. gram_lag_4
  15. z_term_2
  16. d_lag_7
  17. d_lag_24
  18. z_term_3
  19. gram_lag_10
  20. z_gram_lag_6

Rankings combinados:
                correlation  mlp_gevrey  rf_purity  rf_accuracy  mean_rank
d_lag_1                34.0         3.0        3.0          3.0      10.75
z_cogram               53.0         1.0        1.0          1.0      14.00
z_term_2               47.0         4.0        9.0          9.0      17.25
z_cogram_lag_1         54.0         7.0        4.0          4.0      17.25
z_gram                 72.0         2.0        2.0          2.0      19.50
z_gram_lag_1           51.0      